# P2P — Training (SmolVLM-500M + QLoRA r=8 across all 7 LM projections)

**Required Kaggle Dataset attachments** (Add Data → Datasets):
1. `pixels-to-predictions` — the competition data (auto-attached when you fork from the comp).
2. `p2p-code` — your `.py` files (this notebook expects them at `/kaggle/input/p2p-code/`).
3. `smolvlm-500m-instruct` — the base model files (faster + reproducible vs. re-downloading from HF Hub each run).

**Settings panel** (right sidebar):
- Accelerator: **GPU T4 x1** (or P100 if available — slightly faster).
- Internet: **ON** (required if you skip attaching the base model dataset; nice-to-have for `pip install` even if you do attach it).
- Persistence: **Files only** is fine.

**After this notebook finishes:** click *Save Version → Save & Run All*. The trained adapter at `/kaggle/working/outputs/<run_name>/final_adapter/` will be saved as the notebook's output, which you can then attach to the submission notebook as a Dataset.


## 1. Make our `.py` modules importable

In [ ]:
import sys, os
from pathlib import Path

CODE_DIR = '/kaggle/input/p2p-code'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Verify the modules are visible
for f in ['config.py', 'data.py', 'build_model.py', 'scoring.py', 'train.py']:
    assert (Path(CODE_DIR) / f).exists(), f'Missing: {f}'
print('Code modules found.')


## 2. Install / verify dependencies

Kaggle's GPU image has most of what we need; we just pin the versions used in the starter notebook.


In [ ]:
# Pin versions to match the starter notebook for reproducibility.
# Use --quiet so the cell output stays readable.
!pip install --quiet --upgrade transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate


In [ ]:
import torch, transformers, peft, bitsandbytes as bnb
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'peft         : {peft.__version__}')
print(f'bitsandbytes : {bnb.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')


## 3. Patch CFG with Kaggle paths

The default `CFG` uses local paths (`data/`, `outputs/`). We override them for Kaggle.


In [ ]:
from config import CFG

# Competition data — adjust if your attached dataset has a different name.
# Inspect /kaggle/input to confirm the path:
!ls /kaggle/input/


In [ ]:
# Set this to whatever shows up under /kaggle/input/ for the competition.
# When you fork from the competition page, it should be 'pixels-to-predictions'.
DATA_ROOT = Path('/kaggle/input/pixels-to-predictions')

# Some competitions put files in DATA_ROOT directly, others under DATA_ROOT/'data'.
# Auto-detect:
if (DATA_ROOT / 'train.csv').exists():
    CFG.data_dir = DATA_ROOT
elif (DATA_ROOT / 'data' / 'train.csv').exists():
    CFG.data_dir = DATA_ROOT / 'data'
else:
    raise FileNotFoundError(f'train.csv not found under {DATA_ROOT}. Inspect the dataset structure.')

# Point at the LOCAL base model dataset instead of downloading from HF Hub each run.
# This avoids re-downloading ~1GB on every training session, removes a network failure
# mode, and pins the exact model bytes for reproducibility.
BASE_MODEL_DIR = Path('/kaggle/input/smolvlm-500m-instruct')
if BASE_MODEL_DIR.exists():
    CFG.model_id = str(BASE_MODEL_DIR)
    print(f'Using local base model: {CFG.model_id}')
else:
    print(f'WARNING: {BASE_MODEL_DIR} not attached. Will download from HF Hub instead.')
    print(f'  This works but is slower and requires internet. Consider attaching the dataset.')

CFG.output_dir = Path('/kaggle/working/outputs')
CFG.log_dir = Path('/kaggle/working/logs')
CFG.output_dir.mkdir(parents=True, exist_ok=True)
CFG.log_dir.mkdir(parents=True, exist_ok=True)

print(f'data_dir   : {CFG.data_dir}')
print(f'output_dir : {CFG.output_dir}')
print(f'model_id   : {CFG.model_id}')
print(f'run_name   : {CFG.run_name}')
print(f'lora r/alpha : {CFG.lora_r}/{CFG.lora_alpha}')
print(f'epochs     : {CFG.epochs}')
print(f'eff. batch : {CFG.per_device_batch_size * CFG.grad_accum_steps}')


## 4. Smoke test

Always do this before launching real training. Catches bad paths, regex mismatches, OOM, etc.


In [ ]:
# Run the smoke test inline. Takes ~2-5 min on a T4.
import importlib, smoke_test
importlib.reload(smoke_test)  # in case you re-run this cell after editing

# smoke_test reads CFG.data_dir, so we just call its main() with sys.argv override:
import sys as _sys
_old_argv = _sys.argv
_sys.argv = ['smoke_test', '--data_dir', str(CFG.data_dir), '--n_val', '32']
try:
    smoke_test.main()
finally:
    _sys.argv = _old_argv


## 5. Full training

If the smoke test passed, run the real training. With ~10K train examples and 2 epochs at effective batch size 16, expect 4-8 hours on a T4.

**If you're doing an ablation run**, change `CFG.run_name` BEFORE this cell so outputs don't clobber each other.


In [ ]:
# Optional: override run_name for ablations
# CFG.run_name = 'smolvlm_qlora_r8_all7_v2_largerimg'

# Optional: override epochs/lr from the cell
# CFG.epochs = 1            # for a quick first run
# CFG.learning_rate = 1e-4  # for a more conservative run

import importlib, train
importlib.reload(train)

import sys as _sys
_old_argv = _sys.argv
_sys.argv = [
    'train',
    '--data_dir', str(CFG.data_dir),
    '--output_dir', str(CFG.output_dir),
    '--run_name', CFG.run_name,
    '--epochs', str(CFG.epochs),
    '--lr', str(CFG.learning_rate),
    '--lora_r', str(CFG.lora_r),
]
try:
    train.main()
finally:
    _sys.argv = _old_argv


## 6. Verify outputs and clean up unnecessary files

Kaggle limits output size to 20 GB. Trainer writes intermediate checkpoints which we don't need.


In [ ]:
import shutil

run_dir = CFG.output_dir / CFG.run_name
print(f'Contents of {run_dir}:')
!ls -la {run_dir}

# Remove intermediate Trainer checkpoints (we only need final_adapter/).
for ckpt in run_dir.glob('checkpoint-*'):
    print(f'  removing {ckpt.name}')
    shutil.rmtree(ckpt)

print()
print('After cleanup:')
!ls -la {run_dir}
print()
print('Adapter contents:')
!ls -la {run_dir}/final_adapter
print()
print(f'Total output size:')
!du -sh /kaggle/working


## 7. Print the val accuracy result

For your TECH_LOG.md.


In [ ]:
import json as _json

vr = run_dir / 'val_results.json'
if vr.exists():
    print(_json.dumps(_json.loads(vr.read_text()), indent=2))
else:
    print('val_results.json not found — training may have failed.')


## Next steps

1. Click **Save Version → Save & Run All** to commit this run.
2. Once the version is saved, go to *Notebook → Output* and click **New Dataset** to publish `/kaggle/working/outputs/<run_name>/final_adapter/` as a Kaggle Dataset.
3. Open the **submission notebook** and attach (a) the new adapter dataset, and (b) a dataset containing the base SmolVLM model files.
